In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pypdf import PdfReader

sys.path.insert(0, "..")
from schema import PurchaseOrder

load_dotenv(find_dotenv(usecwd=True))
llm = OpenAI(base_url="http://localhost:4000", api_key=os.environ["LITELLM_MASTER_KEY"])
DATA = Path("../data")

In [37]:
def pdf_text(path: Path, mode: str = "layout") -> str:
    return "\n".join(
        page.extract_text(extraction_mode=mode) or "" for page in PdfReader(path).pages
    )


text = pdf_text(DATA / "pos" / "PO-0002.pdf")
i = text.find("Unterleg")
print(text[i - 200 : i + 400])

reis          Gesamtpreis

1                         Rillenkugellager 6205-2RS C3                                   17    PC                    8,75                  148,75

2        400103           Unterlegscheibe DIN 125 A 8,4 verzinkt, 100                    47    PAK                   3,20                  150,40
                          Stk

3                         Hydrauliköl HLP 46                                             20    L                     3,45                   69,00

4        400125           LED-Hallenstrahler 150 W IP65                                  25    PC     


In [17]:
SYSTEM_PROMPT = f"""You extract purchase orders into JSON for Alpentech Industriebedarf GmbH.

Context:
- Alpentech Industriebedarf GmbH is the SUPPLIER that receives these purchase orders.
- The CUSTOMER is the company that issued the purchase order (the buyer).
  Never use Alpentech as the customer.

Field rules:
- customer_name: the buyer's legal company name only.
- ship_to_address: one line in the form "Street Number, PostalCode City, Country".
  Do not include company names, departments (e.g. "Wareneingang") or attention lines.
- description: copy exactly as printed. If a description wraps onto the next line,
  join the parts with a single space.
- Dates as YYYY-MM-DD.
- Numbers as plain decimals with a dot and no thousands separator, e.g. 2060.90.
- Copy text fields exactly as printed. Do not translate.
- If a field is not on the document, use null.
- line_no starts at 1, in document order.

Return only JSON matching this JSON schema: {json.dumps(PurchaseOrder.model_json_schema())}
"""


def extract(text: str, doc_id: str) -> PurchaseOrder:
    response = llm.chat.completions.create(
        model="o2c-extract",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        extra_body={"metadata": {"trace_name": "extract", "tags": ["poc", doc_id]}},
    )
    return PurchaseOrder.model_validate_json(response.choices[0].message.content)

In [18]:
def compare(doc_id: str) -> None:
    result = extract(pdf_text(DATA / "pos" / f"{doc_id}.pdf"), doc_id)
    truth = PurchaseOrder.model_validate_json(
        (DATA / "ground_truth" / f"{doc_id}.json").read_text(encoding="utf-8")
    )
    print(doc_id, "MATCH" if result == truth else "DIFFERENT")
    for field in PurchaseOrder.model_fields:
        if getattr(result, field) != getattr(truth, field):
            print(
                f"  {field}:\n    got:      {getattr(result, field)}\n    expected: {getattr(truth, field)}"
            )


compare("PO-0001")

PO-0001 MATCH


In [19]:
for doc_id in ["PO-0002", "PO-0003", "PO-0004", "PO-0005", "PO-0006"]:
    compare(doc_id)

PO-0002 DIFFERENT
  lines:
    got:      [LineItem(line_no=1, material_number=None, description='Rillenkugellager 6205-2RS C3', quantity=Decimal('17'), unit='PC', unit_price=Decimal('8.75'), line_total=Decimal('148.75')), LineItem(line_no=2, material_number='400103', description='Unterlegscheibe DIN 125 A 8,4 verzinkt, 100', quantity=Decimal('47'), unit='PAK', unit_price=Decimal('3.2'), line_total=Decimal('150.4')), LineItem(line_no=3, material_number=None, description='Hydrauliköl HLP 46', quantity=Decimal('20'), unit='L', unit_price=Decimal('3.45'), line_total=Decimal('69')), LineItem(line_no=4, material_number='400125', description='LED-Hallenstrahler 150 W IP65', quantity=Decimal('25'), unit='PC', unit_price=Decimal('129'), line_total=Decimal('3225')), LineItem(line_no=5, material_number='400112', description='Hochleistungsfett Lithium EP2, 400 g Kartusche', quantity=Decimal('19'), unit='PC', unit_price=Decimal('5.35'), line_total=Decimal('101.65')), LineItem(line_no=6, material_nu

In [38]:
from collections import Counter

from pydantic import ValidationError
from scoring import score


def extract_with_stats(text: str, doc_id: str):
    raw = llm.chat.completions.with_raw_response.create(
        model="o2c-extract",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        extra_body={
            "metadata": {
                "trace_name": "extract",
                "tags": ["poc", "prompt-v3-layout", doc_id],
            }
        },
    )
    response = raw.parse()
    cost = float(raw.headers.get("x-litellm-response-cost") or 0)
    try:
        result = PurchaseOrder.model_validate_json(response.choices[0].message.content)
    except ValidationError:
        result = None
    return result, response.usage.total_tokens, cost

In [39]:
rows = []
for pdf in sorted((DATA / "pos").glob("*.pdf")):
    doc_id = pdf.stem
    truth = PurchaseOrder.model_validate_json(
        (DATA / "ground_truth" / f"{doc_id}.json").read_text(encoding="utf-8")
    )
    result, tokens, cost = extract_with_stats(pdf_text(pdf), doc_id)
    rows.append(
        {"doc_id": doc_id, "tokens": tokens, "cost": cost, **score(result, truth)}
    )
    print(doc_id, rows[-1]["correct"], "/", rows[-1]["total"])

correct = sum(r["correct"] for r in rows)
total = sum(r["total"] for r in rows)
perfect = sum(r["correct"] == r["total"] for r in rows)
print(f"\nField accuracy:   {correct / total:.1%}")
print(f"Perfect POs:      {perfect} / {len(rows)}")
print(f"Invalid outputs:  {sum('invalid output' in r['errors'] for r in rows)}")
print(f"Avg tokens / PO:  {sum(r['tokens'] for r in rows) / len(rows):.0f}")
print(f"Cost / PO:        ${sum(r['cost'] for r in rows) / len(rows):.5f}")
print(
    "Most common errors:", Counter(e for r in rows for e in r["errors"]).most_common(5)
)

PO-0001 32 / 32
PO-0002 44 / 44
PO-0003 20 / 20
PO-0004 32 / 32
PO-0005 44 / 44
PO-0006 20 / 20
PO-0007 38 / 38
PO-0008 14 / 14
PO-0009 20 / 20
PO-0010 50 / 50
PO-0011 56 / 56
PO-0012 50 / 50
PO-0013 56 / 56
PO-0014 32 / 32
PO-0015 56 / 56
PO-0016 25 / 26
PO-0017 38 / 38
PO-0018 14 / 14
PO-0019 26 / 26
PO-0020 56 / 56
PO-0021 50 / 50
PO-0022 44 / 44
PO-0023 56 / 56
PO-0024 20 / 20
PO-0025 44 / 44
PO-0026 32 / 32
PO-0027 50 / 50
PO-0028 26 / 26
PO-0029 50 / 50
PO-0030 26 / 26
PO-0031 38 / 38
PO-0032 20 / 20
PO-0033 50 / 50
PO-0034 26 / 26
PO-0035 56 / 56
PO-0036 38 / 38
PO-0037 20 / 20
PO-0038 50 / 50
PO-0039 32 / 32
PO-0040 56 / 56
PO-0041 20 / 20
PO-0042 26 / 26
PO-0043 20 / 20
PO-0044 20 / 20
PO-0045 56 / 56
PO-0046 50 / 50
PO-0047 44 / 44
PO-0048 20 / 20
PO-0049 38 / 38
PO-0050 32 / 32

Field accuracy:   99.9%
Perfect POs:      49 / 50
Invalid outputs:  0
Avg tokens / PO:  1513
Cost / PO:        $0.00117
Most common errors: [('line.material_number', 1)]


In [40]:
out = Path("../results")
out.mkdir(exist_ok=True)
(out / "prompt-v3-layout.json").write_text(json.dumps(rows, indent=2))

6722

In [32]:
for doc_id in ["PO-0001", "PO-0002", "PO-0009", "PO-0018"]:
    compare(doc_id)

PO-0001 MATCH
PO-0002 DIFFERENT
  lines:
    got:      [LineItem(line_no=1, material_number=None, description='Rillenkugellager 6205-2RS C3', quantity=Decimal('17'), unit='PC', unit_price=Decimal('8.75'), line_total=Decimal('148.75')), LineItem(line_no=2, material_number='400103', description='Unterlegscheibe DIN 125 A 8,4 verzinkt, 100', quantity=Decimal('47'), unit='PAK', unit_price=Decimal('3.2'), line_total=Decimal('150.4')), LineItem(line_no=3, material_number=None, description='Hydrauliköl HLP 46', quantity=Decimal('20'), unit='L', unit_price=Decimal('3.45'), line_total=Decimal('69')), LineItem(line_no=4, material_number='400125', description='LED-Hallenstrahler 150 W IP65', quantity=Decimal('25'), unit='PC', unit_price=Decimal('129'), line_total=Decimal('3225')), LineItem(line_no=5, material_number='400112', description='Hochleistungsfett Lithium EP2, 400 g Kartusche', quantity=Decimal('19'), unit='PC', unit_price=Decimal('5.35'), line_total=Decimal('101.65')), LineItem(line_no=

In [24]:
for doc_id in ["PO-0002", "PO-0009", "PO-0015", "PO-0019", "PO-0040", "PO-0050"]:
    compare(doc_id)

PO-0002 DIFFERENT
  lines:
    got:      [LineItem(line_no=1, material_number=None, description='Rillenkugellager 6205-2RS C3', quantity=Decimal('17'), unit='PC', unit_price=Decimal('8.75'), line_total=Decimal('148.75')), LineItem(line_no=2, material_number='400103', description='Unterlegscheibe DIN 125 A 8,4 verzinkt, 100', quantity=Decimal('47'), unit='PAK', unit_price=Decimal('3.2'), line_total=Decimal('150.4')), LineItem(line_no=3, material_number=None, description='Hydrauliköl HLP 46', quantity=Decimal('20'), unit='L', unit_price=Decimal('3.45'), line_total=Decimal('69')), LineItem(line_no=4, material_number='400125', description='LED-Hallenstrahler 150 W IP65', quantity=Decimal('25'), unit='PC', unit_price=Decimal('129'), line_total=Decimal('3225')), LineItem(line_no=5, material_number='400112', description='Hochleistungsfett Lithium EP2, 400 g Kartusche', quantity=Decimal('19'), unit='PC', unit_price=Decimal('5.35'), line_total=Decimal('101.65')), LineItem(line_no=6, material_nu

In [1]:
print(
    [
        line
        for line in pdf_text(DATA / "pos" / "PO-0002.pdf").splitlines()
        if "Unterleg" in line or "Stk" in line
    ]
)

NameError: name 'pdf_text' is not defined

In [35]:
bad = [r["doc_id"] for r in rows if r["correct"] != r["total"]]
print("Imperfect:", bad)

for doc_id in bad:
    result, _, _ = extract_with_stats(pdf_text(DATA / "pos" / f"{doc_id}.pdf"), doc_id)
    truth = PurchaseOrder.model_validate_json(
        (DATA / "ground_truth" / f"{doc_id}.json").read_text(encoding="utf-8")
    )
    for got, exp in zip(result.lines, truth.lines):
        if " ".join(got.description.split()) != " ".join(exp.description.split()):
            print(
                f"{doc_id} line {got.line_no}\n  got:      {got.description}\n  expected: {exp.description}"
            )

Imperfect: ['PO-0002', 'PO-0009', 'PO-0015', 'PO-0040', 'PO-0050']
PO-0002 line 2
  got:      Unterlegscheibe DIN 125 A 8,4 verzinkt, 100
  expected: Unterlegscheibe DIN 125 A 8,4 verzinkt, 100 Stk
PO-0009 line 1
  got:      Unterlegscheibe DIN 125 A 8,4 verzinkt, 100
  expected: Unterlegscheibe DIN 125 A 8,4 verzinkt, 100 Stk
PO-0015 line 2
  got:      Unterlegscheibe DIN 125 A 8,4 verzinkt, 100
  expected: Unterlegscheibe DIN 125 A 8,4 verzinkt, 100 Stk
PO-0040 line 1
  got:      Unterlegscheibe DIN 125 A 8,4 verzinkt, 100
  expected: Unterlegscheibe DIN 125 A 8,4 verzinkt, 100 Stk
PO-0050 line 2
  got:      Unterlegscheibe DIN 125 A 8,4 verzinkt, 100
  expected: Unterlegscheibe DIN 125 A 8,4 verzinkt, 100 Stk


In [36]:
text = pdf_text(DATA / "pos" / "PO-0002.pdf")
i = text.find("Unterleg")
print(text[i : i + 200])

Unterlegscheibe DIN 125 A 8,4 verzinkt, 100
Stk
47
PAK
3,20
150,40
3
Hydrauliköl HLP 46
20
L
3,45
69,00
4
400125
LED-Hallenstrahler 150 W IP65
25
PC
129,00
3.225,00
5
400112
Hochleistungsfett Lithium 
